# RG3 공정 관계 변화 및 앙상블 고도화
4개 새로운 관계변화 피처, 3모델 및 학습정상 점수척도보정.

In [1]:
from pathlib import Path
import json
from sklearn.base import clone
from sklearn.ensemble import ExtraTreesClassifier,RandomForestClassifier
from catboost import CatBoostClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import confusion_matrix,average_precision_score
root=Path.cwd()
if not (root/'data').exists():root=root.parent
# Rebuild original causal windows only, not the previous sequence learners.
source=''.join(json.loads((root/'notebooks/28_rg3_sequence_ensembles.ipynb').read_text(encoding='utf-8'))['cells'][1]['source'])
exec(source.split('factories=')[0])
available=np.array([r['available_shots'] for r in coverage])
def relation_z(target,predictor):
    ty=target[:-1];px=predictor[:-1]
    good=np.isfinite(ty)&np.isfinite(px)
    if good.sum()<8 or not np.isfinite(target[-1]+predictor[-1]):return np.nan
    ty=ty[good];px=px[good];x0=np.mean(px);y0=np.mean(ty)
    slope=np.sum((px-x0)*(ty-y0))/(np.sum((px-x0)**2)+1e-3)
    residual=ty-(y0+slope*(px-x0))
    scale=max(np.sqrt(np.mean(residual**2)),1.)
    return (target[-1]-(y0+slope*(predictor[-1]-x0)))/scale
phys=[]
for i,window in enumerate(X):
    if available[i]<17:phys.append([np.nan]*4);continue
    p=window[4,-17:];c=window[3,-17:];b=window[5,-17:]
    thermal=window[0,-17:-1]-window[1,-17:-1]
    def slope8(a):
        if not np.isfinite(a).all():return np.nan
        return np.polyfit(np.arange(len(a)),a,1)[0]
    accel=(slope8(thermal[8:])-slope8(thermal[:8]))/max(float(np.nanstd(thermal)),.1)
    dp=np.diff(p[:-1]);dc=np.diff(c[:-1])
    good=np.isfinite(dp)&np.isfinite(dc)
    cosine=np.dot(dp[good],dc[good])/(np.linalg.norm(dp[good])*np.linalg.norm(dc[good])+1e-6)
    phys.append([relation_z(p,c),relation_z(p,b),accel,cosine])
P=np.array(phys);feature_names=['pressure_cushion_local_error_z','pressure_backpressure_local_error_z','thermal_gradient_acceleration','pressure_cushion_delta_cosine']
feature_table=pd.DataFrame(P,columns=feature_names)
feature_table.insert(0,'TimeStamp',d.TimeStamp);feature_table['actual']=y;feature_table['available_shots']=available
feature_table.to_csv(tables/'rg3_relation_dynamics_features.csv',index=False,encoding='utf-8-sig')
learners={'physical_RF':RandomForestClassifier(n_estimators=200,max_depth=4,min_samples_leaf=8,class_weight='balanced_subsample',random_state=42,n_jobs=-1),
'physical_ET':ExtraTreesClassifier(n_estimators=200,max_depth=4,min_samples_leaf=8,class_weight='balanced',random_state=42,n_jobs=-1),
'physical_Cat':CatBoostClassifier(iterations=200,depth=3,learning_rate=.03,l2_leaf_reg=5,auto_class_weights='Balanced',random_seed=42,thread_count=4,verbose=False,allow_writing_files=False)}
pcache={}
def physical_predict(name,ref,q):
    key=(name,tuple(ref),tuple(q))
    if key in pcache:return pcache[key]
    assert set(groups.iloc[ref]).isdisjoint(groups.iloc[q])
    if len(np.unique(y[ref]))<2:out=np.full(len(q),float(y[ref][0]) if len(ref) else 0.)
    elif pd.DataFrame(P[ref]).nunique().le(1).all():out=np.full(len(q),.5)
    else:
        model=make_pipeline(SimpleImputer(strategy='median',keep_empty_features=True),clone(learners[name]))
        out=model.fit(P[ref],y[ref]).predict_proba(P[q])[:,1]
    pcache[key]=out
    return out
def normal_rank(reference,query):
    # Empirical normal-score CDF, fit only on training-inner normal scores.
    reference=np.sort(reference[np.isfinite(reference)])
    if not len(reference):return np.zeros(len(query))
    return np.searchsorted(reference,query,side='left')/len(reference)
records=[]
for scheme in ['leave_one_block_out','past_blocks_only']:
    for j,block in enumerate(blocks):
        q=np.flatnonzero(groups.eq(block));ref=np.flatnonzero(~groups.eq(block)) if scheme=='leave_one_block_out' else np.flatnonzero(groups.isin(blocks[:j]))
        if len(np.unique(y[ref]))<2:continue
        outer=np.column_stack([physical_predict(name,ref,q) for name in learners])
        inner=np.full((len(ref),3),np.nan);rf_inner=np.full(len(ref),np.nan)
        rf_outer=pair_predict(ref,q).max(axis=1)
        for ib in groups.iloc[ref].unique():
            mask=groups.iloc[ref].eq(ib).to_numpy()
            if (~mask).any():
                for k,name in enumerate(learners):inner[mask,k]=physical_predict(name,ref[~mask],ref[mask])
                rf_inner[mask]=pair_predict(ref[~mask],ref[mask]).max(axis=1)
        ranked_outer=np.column_stack([normal_rank(inner[y[ref]==0,k],outer[:,k]) for k in range(3)])
        ranked_inner=np.column_stack([normal_rank(inner[y[ref]==0,k],inner[:,k]) for k in range(3)])
        old_rank_outer=normal_rank(rf_inner[y[ref]==0],rf_outer)
        old_rank_inner=normal_rank(rf_inner[y[ref]==0],rf_inner)
        candidates={name:(outer[:,k],inner[:,k]) for k,name in enumerate(learners)}
        candidates['physical3_mean']=(outer.mean(axis=1),inner.mean(axis=1))
        candidates['physical3_rank_mean']=(ranked_outer.mean(axis=1),ranked_inner.mean(axis=1))
        candidates['RF_physical_rank_mean']=((old_rank_outer+ranked_outer.mean(axis=1))/2,(old_rank_inner+ranked_inner.mean(axis=1))/2)
        candidates['RF_physical_raw_max']=(np.maximum(rf_outer,outer.mean(axis=1)),np.maximum(rf_inner,inner.mean(axis=1)))
        candidates['RF_baseline']=(rf_outer,rf_inner)
        for name,(score,cal) in candidates.items():
            normal=cal[(y[ref]==0)&np.isfinite(cal)]
            for budget in [.01,.03,.05]:
                cut=float(np.quantile(normal,1-budget,method='higher')) if len(normal) else 1.
                for k,z in enumerate(q):records.append(dict(scheme=scheme,model=name,budget=budget,TimeStamp=d.TimeStamp.iloc[z],block=block,actual=int(y[z]),score=float(score[k]),threshold=cut,prediction=int(score[k]>cut)))
        print('Completed dynamics',scheme,block,flush=True)
pp=pd.DataFrame(records);rows=[]
for (scheme,name,budget),g in pp.groupby(['scheme','model','budget'],sort=False):
    tn,fp,fn,tp=confusion_matrix(g.actual,g.prediction,labels=[0,1]).ravel()
    rows.append(dict(scheme=scheme,model=name,budget=budget,shots=len(g),gas=int(g.actual.sum()),AP=average_precision_score(g.actual,g.score),TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn)))
metrics=pd.DataFrame(rows)
metrics.to_csv(tables/'rg3_relation_dynamics_metrics.csv',index=False,encoding='utf-8-sig')
pp.to_csv(tables/'rg3_relation_dynamics_predictions.csv',index=False,encoding='utf-8-sig')
summary=metrics.loc[metrics.budget.eq(.01)].sort_values(['scheme','AP'],ascending=[True,False])
lines=['# RG3 관계변화4피처와 점수척도보정 앙상블','',
'RH 정상566/가스17, 기존관측구간분리/과거평가. 이전16샷 관계를 현재샷에 적용해 pressure-cushion 및 pressure-backpressure 예측오차를 직전오차RMSE(하한1압력단위)로 나눈다. 참조16샷은 정상으로확인된자료가아니므로정상원인모델로해석하지않음.',
'직전16샷의 배럴2-5 온도차 추세가 앞8→뒤8에서 얼마나바뀌는지, pressure/cushion 15개차분의코사인일치도를추가한다. 총4피처. 17샷이력미만은결측으로유지하며모델학습집합에서대체;샷제외하지않음. 미래데이터사용없음.',
'작은RF/ExtraTrees/CatBoost3개고정평균및학습내부정상점수CDF로척도보정한평균. 기존RF두신호결합과추가결합. 정상CDF는각바깥학습내부예측만사용하고평가정답사용금지. CDF의내부정상표본재사용이므로평가오탐률보장아님. 정답기반최적가중치/임계값탐색하지않음.',
'이미반복탐색한동일자료이며AP로후보를고르면최종독립성능아님. 관측구간은실제Run ID아님. 원본센서이력확보조건. 새시계열이력의raw overlap purge는수행하지않아이전28번과검증조건이다름.',
'','| 평가 | 모델/결합 | AP | TP | FP | FN | TN |','|---|---|---:|---:|---:|---:|---:|']
for r in summary.itertuples():lines.append(f'| {r.scheme} | {r.model} | {r.AP:.4f} | {r.TP} | {r.FP} | {r.FN} | {r.TN} |')
(root/'reports/rg3_relation_dynamics_comparison.md').write_text('\n'.join(lines),encoding='utf-8')
print(summary.to_string(index=False));print('Gas with full history',int(((available>=17)&(y==1)).sum()))


             scheme                 model  budget  shots  gas       AP  TP  FP  FN  TN
leave_one_block_out           RF_baseline    0.01    583   17 0.168625   4   6  13 560
leave_one_block_out   RF_physical_raw_max    0.01    583   17 0.066972   2   4  15 562
leave_one_block_out RF_physical_rank_mean    0.01    583   17 0.047240   1   9  16 557
leave_one_block_out           physical_RF    0.01    583   17 0.028529   0   5  17 561
leave_one_block_out   physical3_rank_mean    0.01    583   17 0.028198   0   3  17 563
leave_one_block_out          physical_Cat    0.01    583   17 0.027478   0   4  17 562
leave_one_block_out        physical3_mean    0.01    583   17 0.027137   0   3  17 563
leave_one_block_out           physical_ET    0.01    583   17 0.026552   0   0  17 566
   past_blocks_only           RF_baseline    0.01    440   13 0.081203   2   4  11 423
   past_blocks_only   RF_physical_raw_max    0.01    440   13 0.043311   0   7  13 420
   past_blocks_only   physical3_rank_mean  

In [1]:
# Post-hoc exploration after observing the CatBoost additional gas case. Not independently selected.
p=pd.read_csv(tables/'rg3_relation_dynamics_predictions.csv')
z=p[p.model.eq('RF_baseline')].merge(p[p.model.eq('physical_Cat')],on=['scheme','budget','TimeStamp','block','actual'],validate='one_to_one',suffixes=('_rf','_cat'))
z['union']=z.prediction_rf.eq(1)|z.prediction_cat.eq(1)
or_rows=[]
for (scheme,budget),g in z.groupby(['scheme','budget']):
    tn,fp,fn,tp=confusion_matrix(g.actual,g.union,labels=[0,1]).ravel()
    or_rows.append(dict(scheme=scheme,budget=budget,TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),post_hoc=True))
or_metrics=pd.DataFrame(or_rows)
or_metrics.to_csv(tables/'rg3_relation_dynamics_exploratory_or.csv',index=False,encoding='utf-8-sig')
print(or_metrics.to_string(index=False))


             scheme  budget  TP  FP  FN  TN  post_hoc
leave_one_block_out    0.01   4  10  13 556      True
leave_one_block_out    0.03   4  41  13 525      True
leave_one_block_out    0.05   4  69  13 497      True
   past_blocks_only    0.01   3  15  10 412      True
   past_blocks_only    0.03   4  38   9 389      True
   past_blocks_only    0.05   4  53   9 374      True